# NB_OktaUserforAI_Bronze_To_Silver
**Source:** `AI_Medallion_Bronze.dbo.CoreOktaUser`  
**Target:** `AI_Medallion.Silver.OktaUserforAI`  
**Load type:** Full (overwrite)

In [ ]:
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "common" / "bootstrap.py").is_file():
        os.chdir(_repo)
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")
# After kernel restart: run this cell, bootstrap, then read/transform/write cells.
# read_table() caches parquet under ~/.fabric/medallion_cache/ (Windows: %USERPROFILE%\.fabric\medallion_cache\).
# Re-read from OneLake (ignore cache): os.environ["MEDALLION_CACHE_REFRESH"] = "1"
# Delete ALL local cache: remove the medallion_cache folder (PowerShell: Remove-Item -Recurse -Force $env:USERPROFILE\.fabric\medallion_cache)
# Delete ONE table cache: remove its subfolder (e.g. AI_Medallion__Silver__cursor_usage).
# Disable caching: os.environ["MEDALLION_LOCAL_TABLE_CACHE"] = "0"


In [ ]:
%run ./common/bootstrap


In [3]:
apply_local_merge_spark_conf(spark)
bronze = read_table_broken_lineage(TABLE_OKTA_BRONZE, "okta_bronze")
country_lookup = break_lineage_local(
    load_country_lookup(spark, TABLE_COUNTRY), "country_lookup"
)

login_norm = F.lower(F.trim(F.col("login")))
email_norm = F.lower(F.trim(F.col("email")))

StatementMeta(, acf18394-be6b-474d-877c-79cb39d5ba74, 10, Finished, Available, Finished, False)

In [4]:
okta = bronze.select(
    F.col("id").alias("OktaUserId"),
    F.trim(F.col("display_name")).alias("OktaUserFullName"),
    F.trim(F.col("first_name")).alias("OktaUserFirstName"),
    F.trim(F.col("last_name")).alias("OktaUserLastName"),
    login_norm.alias("OktaUserEmail"),
    email_norm.alias("OktaUserFullEmail"),
    F.trim(F.col("employee_number")).alias("OktaUserEmployeeNumber"),
    F.trim(F.col("region")).alias("OktaUserRegion"),
    F.col("okta_id").alias("OktaUserOktaId"),
    F.col("updated_at").cast("timestamp").alias("OktaUserUpdatedAt"),
    F.col("deleted_at").cast("timestamp").alias("OktaUserDeletedAt"),
    F.trim(F.col("country")).alias("_raw_country"),
    F.trim(F.col("okta_tenant")).alias("OktaUserTenant"),
)

okta = okta.filter(
    F.col("OktaUserId").isNotNull()
    & is_valid_email(F.col("OktaUserEmail"))
    & is_valid_email(F.col("OktaUserFullEmail"))
    & (F.lower(F.col("OktaUserRegion")) != F.lit("japan-onentt"))
    & (F.lower(F.col("OktaUserTenant")) != F.lit("connect-onentt"))
)

okta = with_normalized_country(okta, "_raw_country", country_lookup).drop("_raw_country")

StatementMeta(, acf18394-be6b-474d-877c-79cb39d5ba74, 11, Finished, Available, Finished, False)

In [6]:
write_full_table(okta, TABLE_OKTA_SILVER)

show_sample(okta.limit(10))

StatementMeta(, acf18394-be6b-474d-877c-79cb39d5ba74, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 920fcc02-b78c-4a0f-8960-e6ffe43989b9)